# SDH exp_004 — 변이 유형 기반 조합 실험

exp_003 1위 변이 유형 피처를 기준으로 빈도 필터와 hotspot 크기를 비교합니다.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "common").is_dir() and (p / "experiments").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("프로젝트 루트를 찾지 못했습니다.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from experiments.SDH.exp_004_feature_combinations.run_benchmark import run

## LR seed 42 — 10개 조합 비교

In [ ]:
leaderboard = run()
leaderboard[[
    "preprocessing", "oof_f1_macro_mean", "oof_accuracy_mean",
    "fold_f1_macro_std", "elapsed_seconds",
]]

## LR 3-seed 자동 confirmation

case 01보다 Macro F1이 0.005 이상 높은 후보를 자동 선택하고 기준 후보와 함께 반복 검증합니다.

In [ ]:
reference_case = "case_01_types_reference"
min_f1_improvement = 0.005
reference_f1 = float(
    leaderboard.loc[
        leaderboard["preprocessing"].eq(reference_case),
        "oof_f1_macro_mean",
    ].iloc[0]
)
selection = leaderboard.assign(
    f1_improvement_vs_reference=(
        leaderboard["oof_f1_macro_mean"] - reference_f1
    )
)
selected_cases = selection.loc[
    selection["preprocessing"].ne(reference_case)
    & selection["f1_improvement_vs_reference"].ge(min_f1_improvement),
    "preprocessing",
].tolist()
display(selection[[
    "preprocessing", "oof_f1_macro_mean",
    "f1_improvement_vs_reference",
]])
print(f"자동 선택 후보: {selected_cases}")

if selected_cases:
    confirmed = run(
        selected_cases=[reference_case, *selected_cases],
        confirmation=True,
    )
    display(confirmed)
else:
    print("기준 대비 +0.005 이상 후보가 없어 confirmation을 생략합니다.")

## 선택적 LightGBM 2차 검증

3-seed 결과를 확인한 뒤 안정적인 최고 후보만 직접 입력합니다.

In [ ]:
lgbm_cases = []

if lgbm_cases:
    lgbm_leaderboard = run(
        selected_cases=[reference_case, *lgbm_cases],
        model="lightgbm",
    )
    display(lgbm_leaderboard)
else:
    print("LR confirmation 후 LightGBM 후보를 선택하세요.")